In [5]:
import os
import json
import sqlite3
import chromadb
from logging import exception

#from chromadb.experimental.density_relevance import collection
from groq import Groq
import requests
from numpy.ma.core import empty
from simpleeval import simple_eval
from ddgs import DDGS


In [6]:
client = Groq(api_key=os.environ['llm_key'])

In [45]:
chat_completion = client.chat.completions.create(messages=[
    {
        "role": "user",
        "content": "Today we are creating an agent with ReACT.",
    }
],
    model = "llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


ReACT (Reasoning and Acting in Complex Tasks) is a powerful framework for building intelligent agents. To create an agent with ReACT, we'll need to define its goals, actions, and decision-making processes.

What kind of agent are we trying to create today? Is it for a specific domain, such as robotics, finance, or healthcare? Or are we building a more general-purpose agent?

Also, what's the desired level of complexity for our agent? Are we looking to create a simple reactive agent or a more sophisticated agent that can reason and plan over time? Let me know, and we can start designing our ReACT agent!


In [7]:
def get_weather(location: str) -> str:
    weather_data = {
        "Antalya": "Sunny, 28°C",
        "Karlsruhe": "Rainy, 17°C",
        "Helsinki": "Foggy, 6°C"
    }
    return weather_data.get(location, f"No information found for the city of {location}.")


In [9]:
def get_weather_mock(location: str) -> str:
    weather_data = {"Antalya": "Sunny, 28°C", "Karlsruhe": "Rainy, 17°C"}
    return weather_data.get(location, f"No mock data for {location}.")



In [10]:
def get_weather_withAPI(location: str) -> str:
    try:
        #1.step Trying to get the geocoding
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=en&format=json"
        geo_response = requests.get(geo_url).json()

        if "results"  not in geo_response:
            return "no information found for the city of {location}."

        lat = geo_response["results"][0]["latitude"]
        lon = geo_response["results"][0]["longitude"]

        #2.step Wİth coordinates find the weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url).json()

        current = weather_response["current_weather"]
        temperature = current["temperature"]
        wind_speed = current["windspeed"]


        return f"Current temperature in {location} is {temperature}°C. Wind speed is {wind_speed} km/h."

    except Exception as e:
        # just in case connections is established
        return f"API Error: {str(e)}"









In [11]:
def calculate(answer: str)-> str:
    try:
        answer = answer.strip()
        result = simple_eval(answer)
        return f"Calculated answer: {result}"

    except Exception as e:
        return f"Calculation Error: {str(e)}"



In [12]:
def search_web(query: str)-> str:
    try:
        results =  DDGS().text(query,max_results= 3 )
        if len(results)==0:
             return f"Search engine error: No results found for '{query}'"


        text = ""
        for result in results:
            title = result["title"]
            body = result["body"]
            text  += f"Title: {title}\nbody: {body}\n\n---\n\n"
        return text



    except Exception as e:
        return f"Seach engine error : {str(e)}"







In [13]:
def search_my_documents(query: str) -> str:
    try:
        with open("C:/Agents/yesterday", "r", encoding="utf-8") as file:
            text = file.read()

        if query in text:
            return text
        else:
            return f"The wanted text '{query}' is not found in the documents."
    except Exception as e:
        return f"An error occurred: {str(e)}"

In [14]:
def search_vector_database(query:str)-> str:
    try:
        collec = collection.query(query_texts=[query])
        if  len(collec["documents"][0])>0 :
            searched= collec["documents"][0][0]
            return searched

        else:

             return "The searched term could was not found in the database."

    except Exception as e:
        return f"An error occurred while connecting to the vector database: {str(e)}"



In [15]:
connection = sqlite3.connect("memory.db",check_same_thread=False)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS profile (
 key TEXT PRIMARY KEY,
 value TEXT)
  """)
connection.commit()

In [28]:
api_tools = [
    {

    "type": "function",
    "function": {
        "name": "search_web",
        "description": "Searches the internet for real-time information based on a query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description":"The search query string."
                }
            },
            "required": ["query"]

    }
}
    },
     {

    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Find the current weather information depending on the city name",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description":"The city name  etc. Istanbul."
                }
            },
            "required": ["location"]

    }
}
    },
     {

    "type": "function",
    "function": {
        "name": "calculate",
        "description": "used for  calculating math expressions.",
        "parameters": {
            "type": "object",
            "properties": {
                "answer": {
                    "type": "string",
                    "description":"The math expression to evaluate 25-20."
                }
            },
            "required": ["answer"]

    }
}
    },

     {

    "type": "function",
    "function": {
        "name": "save_memory",
        "description": "Saves the important information based on user query.",
        "parameters": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description":"The term used for memory such as name,age,location."
                },
                "value": {
                    "type": "string",
                    "description":"The term that is the value for the key such as Ali,31,Karlsruhe."
                }
            },
            "required": ["key", "value"]

    }
}
    },
    {

    "type": "function",
    "function": {
        "name": "get_user_profile",
        "description": "Gets the user data from database",
        "parameters": {
            "type": "object",
            "properties": {
                "key": {
                    "type": "string",
                    "description":"The information to look up, such as 'name'."
                },

            },
            "required": ["key"]

    }
}
    },

    {



"type": "function",
    "function": {
        "name": "search_my_documents",
        "description": "Searches files to find a specific info from that text file.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description":"Helps you to find that if a specific info about  that file ."
                },

            },
            "required": ["query"]

    }

}

    },
     {
    "type": "function",
    "function": {
        "name": "search_vector_database",
        "description": "Searches the user's personal documents, past notes, memories, and general knowledge database. ALWAYS use this tool to find information about past events (like what the user ate), passwords, favorite items, or any saved context that is not just a simple key-value profile setting.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query in natural language to find the relevant memory or document (e.g., 'favorite console', 'office password', 'what did I eat yesterday')."
                }
            },
            "required": ["query"]
        }
    }
}



]

In [17]:
def save_memory(key:str,value:str) ->str:
     try:

         cursor.execute("INSERT OR REPLACE INTO profile(key,value) VALUES (?,?)",(key,value))
         connection.commit()
         return f"Successfully saved {key} and {value} in memory."
     except Exception as e:
         return f"A problem occured while saving{str(e)}"




In [18]:
def get_user_profile(key:str)->str:
    try:
        cursor.execute("SELECT value FROM profile WHERE key = ?",(key,))
        value = cursor.fetchone()

        if value:
            return f"Data found in memory: {key} = {value[0]}"
        else:

            return f"No data found in memory for the key: '{key}'. Try a different key."

    except Exception as e:
        return f"An error occured while reading the memory: {str(e)}"

In [19]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="agentic_databank")

collection.add(
    documents=[
        "Last night I ate chiken wings at it was delicous I bought it from Lidl.",
        "The password for the office door is 2099.",
        "my favourite console is playstation 2. "
    ],
    ids=["id1","id2",",id3"]

)

print("vector dataset is loaded")

vector dataset is loaded


In [20]:
system_prompt = """
You are an advanced AI assistant with access to several tools.
Think logically step-by-step to answer the user's questions.
If you need information you don't have, use the appropriate tool.
Once you have all the information you need, provide a clear, helpful final answer.
"""

In [21]:
def run_reAct_agent(user_query:str,tools:dict,max_steps:int = 5 ):
    messages = [


            {"role": "system","content":system_prompt},
            {"role": "user","content":user_query}

    ]

    for step in range(max_steps):
        print(f"\n--- Adım {step + 1} ---")

        response = client.chat.completions.create(
            messages=messages,
            model="openai/gpt-oss-120b",
            temperature=0.1,
            tools = api_tools,
            tool_choice = "auto",
            parallel_tool_calls=True
        )
        response_message = response.choices[0].message


        # 1 Add model's msg to the past msgs(needs to be done)
        messages.append(response_message)

        # 2 final cevaba ulaşma kontrolü
        if response_message.tool_calls:
            print("\n[System]Model wants to use tools")

            for tool_call in response_message.tool_calls:
                action = tool_call.function.name
                action_input = tool_call.function.arguments

                print(f"The chosen tool is {action}")
                print(f"The chosen action is {action_input}")

                # convert JSON to python lib so we can use funcs
                parameters = json.loads(action_input)
                print(f"The chosen parameters is {parameters}")

                if action in tools:
                    observation = tools[action](**parameters)
                    print(f"The chosen action is {observation}")

                    messages.append({

                        "role":"tool",
                        "tool_call_id":tool_call.id,
                        "name":action,
                        "content": str(observation)

                    }
                    )
                else:
                    print(f"The chosen tool is not found  {action}")

        else:
            final_answer = response_message.content
            print(f"\nFinal Answer: {final_answer}")
            return final_answer










In [26]:

my_tools_real = {
    "get_weather": get_weather_withAPI,
    "calculate": calculate,
    "search_web": search_web,
    "save_memory": save_memory,
    "get_user_profile": get_user_profile,
    "search_vector_database": search_vector_database

}


In [23]:
print("\nANSWER:",run_reAct_agent(" what is the weather for Karlsruhe",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is get_weather
The chosen action is {"location":"Karlsruhe"}
The chosen parameters is {'location': 'Karlsruhe'}
The chosen action is Current temperature in Karlsruhe is 28.9°C. Wind speed is 12.9 km/h.

--- Adım 2 ---

Final Answer: The current weather in Karlsruhe is **28.9 °C** with a wind speed of **12.9 km/h**.

ANSWER: The current weather in Karlsruhe is **28.9 °C** with a wind speed of **12.9 km/h**.


In [59]:
print("\nANSWER:",run_reAct_agent(" what is the sqaure root of 16",tools=my_tools_real))


--- Adım 1 ---

Final Answer: The square root of 16 is 4.

ANSWER: The square root of 16 is 4.


In [60]:
print("\nANSWER:",run_reAct_agent(" what is the retail price of Gta 6 for Playtation 5",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is search_web
The chosen action is {"query":"GTA 6 PlayStation 5 retail price"}
The chosen parameters is {'query': 'GTA 6 PlayStation 5 retail price'}
The chosen action is Title: GTA 6 vorbestellen: Preise, Editionen & Release-Termin im Überblick
body: 2 weeks ago - GTA 6 kann ab dem 25. Juni 2026 vorbestellt werden. Alle Infos zu den Preisen, Editionen, PS5 Bundle und dem Release am 19. November 2026. Dieser Beitrag wird fortlaufend aktualisiert!

---

Title: Grand Theft Auto 6 price finally announced for US customers, and later revealed for UK and European customers.
body: 2 weeks ago - The standard version of GTA 6 will retail for $79.99, and be available on Xbox Series X/S and PlayStation 5.

---

Title: r/PS5 on Reddit: Grand Theft Auto 6 will retail for $80 and the Ultimate Edition will be $100
body: 2 weeks ago - GTA 6 price listed as $115 via one retailer as pre-order rumors circulate, and fans are praying it's j

In [61]:
print("\nANSWER:",run_reAct_agent(" my name is Ali  and my favourite game is GTA 6",tools=my_tools_real))



--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is save_memory
The chosen action is {"key":"name","value":"Ali"}
The chosen parameters is {'key': 'name', 'value': 'Ali'}
The chosen action is Successfully saved name and Ali in memory.

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is save_memory
The chosen action is {"key":"favourite_game","value":"GTA 6"}
The chosen parameters is {'key': 'favourite_game', 'value': 'GTA 6'}
The chosen action is Successfully saved favourite_game and GTA 6 in memory.

--- Adım 3 ---

Final Answer: Got it, Ali! I’ve saved your name and noted that your favorite game is GTA 6. If there’s anything else you’d like to share or ask about, just let me know!

ANSWER: Got it, Ali! I’ve saved your name and noted that your favorite game is GTA 6. If there’s anything else you’d like to share or ask about, just let me know!


In [62]:
print("\nANSWER:",run_reAct_agent(" what is my name and my favourite game ",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is get_user_profile
The chosen action is {"key":"name"}
The chosen parameters is {'key': 'name'}
The chosen action is Data found in memory: name = Ali

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is get_user_profile
The chosen action is {"key":"favourite game"}
The chosen parameters is {'key': 'favourite game'}
The chosen action is Data found in memory: favourite game = GTA 6

--- Adım 3 ---

Final Answer: Your name is **Ali**, and your favorite game is **GTA 6**.

ANSWER: Your name is **Ali**, and your favorite game is **GTA 6**.


In [21]:
print("\nANSWER:",run_reAct_agent("look through my files and tell me what I did my files name is yesterday  and tell me what I did ",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is search_my_documents
The chosen action is {"query":"yesterday"}
The chosen parameters is {'query': 'yesterday'}
The chosen action is An error occurred: [Errno 2] No such file or directory: 'C:/Agents/26-07-07.txt'

--- Adım 2 ---

[System]Model wants to use tools
The chosen tool is search_my_documents
The chosen action is {"query":"yesterday.txt"}
The chosen parameters is {'query': 'yesterday.txt'}
The chosen action is An error occurred: [Errno 2] No such file or directory: 'C:/Agents/26-07-07.txt'

--- Adım 3 ---

Final Answer: I’m sorry, but I wasn’t able to locate a file named **“yesterday”** (or any similarly‑named document) in your stored files, so I can’t retrieve its contents for you. If you can provide the exact filename (including its extension) or upload the file again, I’ll be happy to look through it and let you know what you wrote about.

ANSWER: I’m sorry, but I wasn’t able to locate a file named **“yeste

In [30]:
print("\nANSWER:",run_reAct_agent("what did I ate last night  ?",tools=my_tools_real))


--- Adım 1 ---

[System]Model wants to use tools
The chosen tool is search_vector_database
The chosen action is {"query":"what did I eat last night"}
The chosen parameters is {'query': 'what did I eat last night'}
The chosen action is Last night I ate chiken wings at it was delicous I bought it from Lidl.

--- Adım 2 ---

Final Answer: Based on the information stored in your personal records, you mentioned that last night you ate **chicken wings** (which you described as delicious) that you bought from Lidl.

ANSWER: Based on the information stored in your personal records, you mentioned that last night you ate **chicken wings** (which you described as delicious) that you bought from Lidl.
